# Product Segmentation with K-Means Clustering

**Retail supermarket "SWALAYAN KEADILAN" - 2025 POS receipts**

This notebook builds an end-to-end, reproducible machine-learning pipeline that
segments products by their sales behaviour using K-Means clustering. The heavy
lifting lives in the `src/` package; this notebook orchestrates the phases and
presents the results.

## 1. Introduction

Retailers carry thousands of products with very different sales dynamics. Grouping
them into a small number of behavioural segments supports inventory planning,
shelf placement, promotions, and pricing. We use **unsupervised** learning
(K-Means) because there are no pre-existing labels; the goal is interpretability,
not prediction.

In [ ]:
# Make the `src` package importable regardless of the working directory.
import sys
from pathlib import Path

root = Path.cwd()
while not (root / "src").exists() and root != root.parent:
    root = root.parent
sys.path.insert(0, str(root))

import pandas as pd
from IPython.display import Image, display

from src import config, parser, preprocessing, feature_engineering, visualization
from src.clustering import (
    scale_features, evaluate_k, select_optimal_k, fit_kmeans,
    build_cluster_profiles, run_clustering, run_segmented_clustering,
)

pd.set_option("display.float_format", lambda v: f"{v:,.2f}")
config.ensure_output_dirs()
print("Project root:", root)

## 2. Problem Statement

Given raw Indonesian POS receipt logs, segment products into behavioural groups
using K-Means so the business can distinguish fast-moving, slow-moving, and
premium products and act on each group differently.

## 3. Dataset Description

* **Source:** `dataset/struk penjualan 2025/*.TXT` - 209 daily POS receipt logs.
* **Language / currency:** Indonesian; Indonesian Rupiah (IDR). Amounts use `.`
  as a thousands separator (`5.000` = 5000).
* **Receipt structure:** a transaction header line carries the datetime and
  receipt code; each product is a name line plus a quantity line; optional
  `Potongan` rows carry discounts; a trailing `**` marks excise (cukai) goods.

See `SPEC.md` for the full grammar and schema.

## 4. Data Parsing

Parse every receipt file into one row per product line.

In [ ]:
transactions_raw = parser.parse_all()
print(f"{transactions_raw['source_file'].nunique()} files parsed")
print(f"{len(transactions_raw):,} product lines, "
      f"{transactions_raw['transaction_uid'].nunique():,} transactions, "
      f"{transactions_raw['product_code'].nunique():,} distinct products")
transactions_raw.head()

## 5. Data Cleaning

Remove duplicates, drop rows with missing/invalid fields, and validate
quantities and prices.

In [ ]:
transactions, report = preprocessing.clean_transactions(transactions_raw)
print(report.summary())
transactions.to_csv(config.TRANSACTIONS_CSV, index=False)
transactions.describe(include="all").T

## 6. Exploratory Data Analysis

Dataset dimensions, top products, and the distribution / correlation of the
sales features. Figures are written to `reports/figures/`.

In [ ]:
products_eda = feature_engineering.build_product_features(transactions)
print("Products:", len(products_eda))
print("\nTop 10 products by revenue:")
display(products_eda.nlargest(10, "total_revenue")[
    ["product_name", "total_quantity_sold", "total_revenue", "transaction_count"]
])
display(Image(filename=str(visualization.plot_top_products_by_revenue(products_eda))))
display(Image(filename=str(visualization.plot_top_products_by_quantity(products_eda))))
display(Image(filename=str(visualization.plot_distributions(products_eda))))
display(Image(filename=str(visualization.plot_correlation_heatmap(products_eda))))

## 7. Feature Engineering

Aggregate to one row per product and derive ratio features. Alongside the
volume/value features we engineer temporal features (`active_days`,
`monthly_cv`, `recency_days`, `weekend_ratio`) so the model can separate steady
sellers from spiky / seasonal products. The clustering feature set (numerical
only) is defined in `config.CLUSTERING_FEATURES`.

In [ ]:
products = feature_engineering.build_product_features(transactions)
products.to_csv(config.PRODUCTS_CSV, index=False)
print("Clustering features:", config.CLUSTERING_FEATURES)
products.head()

## 8. Data Scaling

Retail features are strongly right-skewed, so we apply `log1p` and then
`StandardScaler`. The resulting matrix is standardised with no missing values.

In [ ]:
import numpy as np

scaled, scaler = scale_features(products)
print("Scaled matrix shape:", scaled.shape)
print("All finite (no NaN/inf):", bool(np.isfinite(scaled).all()))
display(Image(filename=str(visualization.plot_feature_boxplots(products))))

## 9. Elbow Method

Sweep k and inspect the within-cluster sum of squares (inertia).

In [ ]:
scores = evaluate_k(scaled)
display(scores)
display(Image(filename=str(visualization.plot_elbow_and_silhouette(scores))))

## 10. Silhouette Analysis

The silhouette score gives a single objective criterion for choosing k. We pick
the k with the highest score.

In [ ]:
optimal_k = select_optimal_k(scores)
print(f"Optimal k = {optimal_k} "
      f"(silhouette = {scores['silhouette'].max():.3f})")

## 11. K-Means Clustering

The elbow and silhouette curves above guide the choice of k. The final model
matches the pipeline: when `config.SEPARATE_EXCISE` is set, excise (cukai) goods
are scaled and clustered in a separate pass so they do not skew the general
segments. The total cluster count therefore combines both groups, each with its
own silhouette-selected k.

In [ ]:
result = (
    run_segmented_clustering(products)
    if config.SEPARATE_EXCISE
    else run_clustering(products)
)
clustered = result.products
clustered.to_csv(config.CLUSTERED_CSV, index=False)
print(f"Total clusters: {result.optimal_k}")
clustered.groupby("segment_group")["cluster"].nunique().rename("clusters")

## 12. Cluster Visualization

Project to 2-D with PCA and show cluster sizes.

In [ ]:
display(Image(filename=str(visualization.plot_pca_clusters(clustered))))
display(Image(filename=str(visualization.plot_cluster_sizes(clustered))))

## 13. Cluster Profiling

Average behaviour per cluster (with its `segment_group`), used to label each
segment.

In [ ]:
profiles = result.profiles
profiles.to_csv(config.CLUSTER_PROFILE_CSV, index=False)
profiles

## 14. Business Insights

Read the profile table above and label each cluster, for example:

* **Fast-moving staples** - high quantity, high transaction count, low price.
  Keep deep stock, place at eye level, avoid stock-outs.
* **Premium / high-ticket** - high revenue per transaction, higher price, lower
  volume. Protect margin, feature in bundles, targeted promotions.
* **Long-tail / slow-movers** - low volume and revenue. Review shelf space,
  consider clearance or delisting.
* **Excise (cukai) segment** - cigarettes and similar; clustered separately
  (`segment_group == "excise"`) so their high price/revenue does not distort the
  general segments. Manage for compliance and margin.

Concrete recommendations span inventory management, promotion strategy, shelf
placement, product bundling, and stock optimisation. The narrative version lives
in `reports/final_report.md`.

In [ ]:
# Programmatic helper: rank clusters by total revenue for the write-up.
summary = profiles[["cluster", "segment_group", "n_products",
                    "avg_quantity_sold", "avg_revenue", "avg_price",
                    "revenue_share"]]
summary

## 15. Conclusion

We parsed 209 raw POS receipt logs into a clean transaction table, engineered
product-level features, selected k via the elbow and silhouette methods, and
trained a reproducible K-Means model (`random_state=42`). The resulting segments
are interpretable and map directly to inventory, pricing, and promotion actions.

Re-run the whole pipeline headlessly with:

```
uv run python -m src.run_pipeline
```